###DimUser  AutoLoad

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os
import sys
project_path = os.path.join(os.getcwd(),'..','..')
sys.path.append(project_path)
from utils.transformations import reusable 

In [0]:
df_user = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", "abfss://silver@datalakedeproject.dfs.core.windows.net/DimUser/schema")
    .option("schemaEvolutionMode", "addNewColumns")
    .load("abfss://bronze@datalakedeproject.dfs.core.windows.net/DimUser"))



In [0]:
df_user = df_user.withColumn("user_name", upper(col("user_name")))

In [0]:
df_user_obj = reusable()
df_user = df_user_obj.dropColumns(df_user, ["_rescued_data"])
df_user = df_user.dropDuplicates(['user_id'])

In [0]:
df_user.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@datalakedeproject.dfs.core.windows.net/DimUser/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@datalakedeproject.dfs.core.windows.net/DimUser/data")\
    .toTable("spotify_cata.silver.DimUser")


###DimArtist  AutoLoad

In [0]:
df_artist = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", "abfss://silver@datalakedeproject.dfs.core.windows.net/DimArtist/schema")
    .option("schemaEvolutionMode", "addNewColumns")
    .load("abfss://bronze@datalakedeproject.dfs.core.windows.net/DimArtist"))



In [0]:
df_artist = df_artist.withColumn("artist_name", upper(col("artist_name")))
df_artist_obj = reusable()
df_artist = df_artist_obj.dropColumns(df_artist, ["_rescued_data"])
df_artist = df_artist.dropDuplicates(['artist_id'])



In [0]:
df_artist.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@datalakedeproject.dfs.core.windows.net/DimArtist/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@datalakedeproject.dfs.core.windows.net/DimArtist/data")\
    .toTable("spotify_cata.silver.DimArtist")

### DimTrack

In [0]:
df_track = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", "abfss://silver@datalakedeproject.dfs.core.windows.net/DimTrack/schema")
    .option("schemaEvolutionMode", "addNewColumns")
    .load("abfss://bronze@datalakedeproject.dfs.core.windows.net/DimTrack"))



In [0]:
df_track = df_track.withColumn("duration_flag", when(col("duration_sec") < 150, "low")\
    .when(col("duration_sec") < 300 , "Medium")\
    .otherwise("short"))
df_track_obj = reusable()
df_track = df_track_obj.dropColumns(df_track, ["_rescued_data"])

df_track = df_track.withColumn("track_name", regexp_replace(col("track_name"), "-", " "))


In [0]:
df_track.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@datalakedeproject.dfs.core.windows.net/DimTrack/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@datalakedeproject.dfs.core.windows.net/DimTrack/data")\
    .toTable("spotify_cata.silver.DimTrack")

### DimDate Autoloader


In [0]:
df_date = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", "abfss://silver@datalakedeproject.dfs.core.windows.net/DimDate/schema")
    .option("schemaEvolutionMode", "addNewColumns")
    .load("abfss://bronze@datalakedeproject.dfs.core.windows.net/DimDate"))



In [0]:
df_date = reusable().dropColumns(df_date, ['_rescued_data'])


In [0]:
df_date.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@datalakedeproject.dfs.core.windows.net/DimDate/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@datalakedeproject.dfs.core.windows.net/DimDate/data")\
    .toTable("spotify_cata.silver.DimDate")

####FactStream Autoload

In [0]:
df_factstream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", "abfss://silver@datalakedeproject.dfs.core.windows.net/FactStream/schema")
    .option("schemaEvolutionMode", "addNewColumns")
    .load("abfss://bronze@datalakedeproject.dfs.core.windows.net/FactStream"))



In [0]:
df_factstream = reusable().dropColumns(df_factstream, ["_rescued_data"])



In [0]:
df_factstream.writeStream.format("delta")\
    .outputMode("append")\
    .option("checkpointLocation", "abfss://silver@datalakedeproject.dfs.core.windows.net/FactStream/checkpoint")\
    .trigger(once=True)\
    .option("path", "abfss://silver@datalakedeproject.dfs.core.windows.net/FactStream/data")\
    .toTable("spotify_cata.silver.FactStream")